In [ ]:
https://colab.research.google.com/drive/1v3iqmoHxvcihzNlysVmVlmAqkZy8T3tc#scrollTo=EHQp85RffhMW

この内容をローカルで実行したい。
ただし、テストはメタデータフィルターでやりたいので以下を良い感じに流用する。


SAMPLE_TEXTS_WITH_METADATA = [
    {
        "text": (
            "今日は素晴らしい一日でした。朝に近所の公園を散歩し、桜が満開で癒されました。"
        ),
        "metadata": {
            "full_doc_id": "doc-park",
            "chunk_order_index": 0,
            "topic": "outdoor",
            "mood": "relaxed",
            "keywords": ["散歩", "公園"],
        },
    },
    {
        "text": (
            "仕事で取り組んでいたAIプロジェクトが完了し、チーム全員で大きな達成感を味わいました。"
        ),
        "metadata": {
            "full_doc_id": "doc-project",
            "chunk_order_index": 1,
            "topic": "work",
            "mood": "達成",
            "keywords": ["仕事", "プロジェクト"],
        },
    },
    {
        "text": (
            "週末に初めてパスタを一から作り、苦労しながらもコクのあるカルボナーラが完成しました。"
        ),
        "metadata": {
            "full_doc_id": "doc-cooking",
            "chunk_order_index": 2,
            "topic": "cooking",
            "mood": "挑戦",
            "keywords": ["料理", "パスタ"],
        },
    },
    {
        "text": (
            "村上春樹の小説を読み進めながら、深層学習の技術書で理論も学んでいます。"
        ),
        "metadata": {
            "full_doc_id": "doc-reading",
            "chunk_order_index": 3,
            "topic": "reading",
            "mood": "集中",
            "keywords": ["読書", "本"],
        },
    },
    {
        "text": (
            "友人と映画館で『君の名は。』を鑑賞し、感動的なストーリーに胸が熱くなりました。"
        ),
        "metadata": {
            "full_doc_id": "doc-movie",
            "chunk_order_index": 4,
            "topic": "movie",
            "mood": "感動",
            "keywords": ["映画"],
        },
    },
]


async def _sample_insert_texts(rag_instance: _SampleRAG):
    for item in SAMPLE_TEXTS_WITH_METADATA:
        await rag_instance.ainsert(item["text"].strip(), metadata=item["metadata"])


async def _prepare_sample_rag() -> _SampleRAG:
    rag = _SampleRAG()
    await _sample_insert_texts(rag)
    return rag


async def _run_sample_queries(
    rag_instance: _SampleRAG,
    queries: list[str],
    metadata_filters: dict[str, str],
):
    modes = ["naive", "mini", "light"]
    results: dict[str, dict[str, list[str]]] = {}

    for query in queries:
        mode_results: dict[str, list[str]] = {}
        for mode in modes:
            answer = await rag_instance.aquery(
                query,
                param=QueryParam(
                    mode=mode,
                    metadata_filters=metadata_filters,
                    only_need_context=True,
                ),
            )
            mode_results[mode] = answer
        results[query] = mode_results

    return results


def test_sample_rag_filters_by_full_doc_id():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"full_doc_id": "doc-movie"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("映画について教えて", param=param))

    assert len(results) == 1
    assert "映画" in results[0]


def test_sample_rag_filters_by_multiple_conditions():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "cooking", "mood": "挑戦"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("料理で何を作りましたか？", param=param))

    assert len(results) == 1
    assert "カルボナーラ" in results[0]


def test_sample_rag_filters_accept_stringified_numbers():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"chunk_order_index": "2"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("料理で何を作りましたか？", param=param))

    assert len(results) == 1
    assert "パスタ" in results[0]


def test_sample_rag_filters_no_match_returns_empty_list():
    rag = asyncio.run(_prepare_sample_rag())
    param = QueryParam(
        mode="naive",
        metadata_filters={"topic": "movie", "mood": "集中"},
        only_need_context=True,
    )
    results = asyncio.run(rag.aquery("読んでいる本について教えて", param=param))

    assert results == []


def test_run_sample_queries_applies_filters_to_each_mode():
    rag = asyncio.run(_prepare_sample_rag())
    metadata_filters = {"topic": "movie"}
    queries = [
        "映画について教えて",
        "散歩について詳しく教えて",
    ]

    results = asyncio.run(_run_sample_queries(rag, queries, metadata_filters))


In [33]:
!nvidia-smi

Fri Oct 24 12:49:20 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 528.24       Driver Version: 528.24       CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name            TCC/WDDM | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ... WDDM  | 00000000:01:00.0  On |                  Off |
|  0%   36C    P2    60W / 450W |    681MiB / 24564MiB |      2%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
!uv run python --version

Python 3.12.11


In [ ]:
# インストールしていく
!uv add pip

In [18]:
!uv add python-dotenv json_repair rouge numpy pandas tiktoken nltk pipmaster

Resolved 175 packages in 775ms
Prepared 3 packages in 1.48s
Uninstalled 1 package in 1ms
Installed 3 packages in 10ms
 + ascii-colors==0.11.4
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + pipmaster==1.0.9


In [16]:
!uv add sentence_transformers

Resolved 173 packages in 1.03s
Prepared 13 packages in 25.02s
Uninstalled 1 package in 2ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 16 packages in 8.43s
 + filelock==3.20.0
 + fsspec==2025.9.0
 + huggingface-hub==0.36.0
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + mpmath==1.3.0
 + networkx==3.5
 + pillow==12.0.0
 + safetensors==0.6.2
 + scikit-learn==1.7.2
 + scipy==1.16.2
 + sentence-transformers==5.1.2
 + sympy==1.14.0
 + threadpoolctl==3.6.0
 + tokenizers==0.22.1
 + torch==2.9.0
 + transformers==4.57.1


In [22]:
!uv add openai tenacity

Resolved 183 packages in 159ms
Prepared 1 package in 1.22s
Uninstalled 1 package in 1ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 37ms
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)
 + tenacity==9.1.2


In [ ]:
# GPUを認識させる
!uv remove torch torchvision torchaudio

!uv run python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [1]:
# 現在のPyTorchを完全にアンインストール
!uv run python -m pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.9.0
Uninstalling torch-2.9.0:
  Successfully uninstalled torch-2.9.0
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121


Uninstalled 2 packages in 2.53s
Installed 1 package in 10.32s


In [2]:
# CUDA版を再インストール
!uv run python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.1-py3-none-any.whl (6.2 MB)

  Attempting uninstall: sympy

    Found existing installation: sympy 1.14.0

   ---------------------------------------- 0/4 [sympy]
    Uninstalling sympy-1.14.0:
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
      Successfully uninstalled sympy-1.14.0
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
   -------------------

Installed 1 package in 14.83s
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
minirag-hku 0.0.2 requires torch[cuda]>=2.8.0, but you have torch 2.5.1+cu121 which is incompatible.


In [1]:
!uv sync

Resolved 184 packages in 474ms
Prepared 1 package in 2.17s
Uninstalled 1 package in 1ms
Installed 1 package in 16ms
 ~ minirag-hku==0.0.2 (from file:///C:/Users/kbpsh/OneDrive/development/project/minirag_dayo)


In [4]:
!uv run python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-win_amd64.whl (6.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (4.1 MB)
  Using cached https://download.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-win_amd64.whl (2449.3 MB)
  Using cached https://download.pytorch.org/whl/sympy-1.13.1-py3-none-any.whl (6.2 MB)

  Attempting uninstall: sympy

    Found existing installation: sympy 1.14.0

   ---------------------------------------- 0/4 [sympy]
    Uninstalling sympy-1.14.0:
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
      Successfully uninstalled sympy-1.14.0
   ---------------------------------------- 0/4 [sympy]
   ---------------------------------------- 0/4 [sympy]
   -------------------

In [5]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device count:', torch.cuda.device_count())
    print('Current device:', torch.cuda.current_device())
    print('Device name:', torch.cuda.get_device_name(torch.cuda.current_device()))

PyTorch version: 2.9.0+cpu
CUDA available: False


In [6]:
!python -c "import sys, torch; print('python exe:', sys.executable); print('python version:', sys.version.splitlines()[0]); print('torch.__version__:', torch.__version__); print('torch.version.cuda:', torch.version.cuda); print('torch.cuda.is_available():', torch.cuda.is_available())"

python exe: C:\Users\kbpsh\OneDrive\development\project\minirag_dayo\.venv\Scripts\python.exe
python version: 3.12.11 (main, Jun 26 2025, 21:17:44) [MSC v.1944 64 bit (AMD64)]
torch.__version__: 2.5.1+cu121
torch.version.cuda: 12.1
torch.cuda.is_available(): True


In [7]:
!uv run -- python -c "import sys, torch; print(sys.executable, torch.__version__, torch.version.cuda, torch.cuda.is_available())"

C:\Users\kbpsh\OneDrive\development\project\minirag_dayo\.venv\Scripts\python.exe 2.9.0+cpu None False


Uninstalled 2 packages in 2.92s
Installed 2 packages in 7.25s


In [ ]:
↓

In [ ]:
!uv lock --upgrade

In [ ]:
!uv sync

In [10]:
# feature-metadata-filtering ブランチでOK
!git branch

* feature-metadata-filtering
  main
  metadata-filter


In [24]:
# 必要なライブラリのインポート
import os
import tempfile
from minirag import MiniRAG, QueryParam
from minirag.llm.hf import (
    hf_model_complete,
    hf_embed,
)
# from minirag.llm.openai import openrouter_openai_complete
from minirag.llm.openai import openai_complete_if_cache
from minirag.utils import EmbeddingFunc
from minirag.utils import (
    wrap_embedding_func_with_attrs,
    locate_json_string_body_from_string,
    safe_unicode_decode,
    logger,
)
from transformers import AutoModel, AutoTokenizer
import asyncio
import warnings
warnings.filterwarnings('ignore')

## 環境変数

In [29]:
import os
from dotenv import load_dotenv

# .envファイルを読み込む
load_dotenv()

# 環境変数を取得する
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')


os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY


print(f"GEMINI_API_KEY: {GEMINI_API_KEY[:5]}********************")
print(f"HF_TOKEN: {HF_TOKEN[:5]}********************")
print(f"OPENROUTER_API_KEY: {OPENROUTER_API_KEY[:5]}********************")

GEMINI_API_KEY: AIzaS********************
HF_TOKEN: hf_UK********************
OPENROUTER_API_KEY: sk-or********************


## セッティング

In [2]:
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
device = "cuda" if torch.cuda.is_available() else "cpu"
# model = SentenceTransformer("cl-nagoya/ruri-v3-30m", device=device)

# Ruri v3 employs a 1+3 prefix scheme to distinguish between different types of text inputs:
# "" (empty string) is used for encoding semantic meaning.
# "トピック: " is used for classification, clustering, and encoding topical information.
# "検索クエリ: " is used for queries in retrieval tasks.
# "検索文書: " is used for documents to be retrieved.
sentences = [
    "川べりでサーフボードを持った人たちがいます",
    "サーファーたちが川べりに立っています",
    "トピック: 瑠璃色のサーファー",
    "検索クエリ: 瑠璃色はどんな色？",
    "検索文書: 瑠璃色（るりいろ）は、紫みを帯びた濃い青。名は、半貴石の瑠璃（ラピスラズリ、英: lapis lazuli）による。JIS慣用色名では「こい紫みの青」（略号 dp-pB）と定義している[1][2]。",
]

embeddings = model.encode(sentences, convert_to_tensor=True)
print(embeddings.size())
# [5, 256]

similarities = F.cosine_similarity(embeddings.unsqueeze(0), embeddings.unsqueeze(1), dim=2)
print(similarities)
# [[1.0000, 0.9540, 0.8512, 0.7322, 0.7274],
#  [0.9540, 1.0000, 0.8531, 0.7437, 0.7305],
#  [0.8512, 0.8531, 1.0000, 0.8910, 0.8649],
#  [0.7322, 0.7437, 0.8910, 1.0000, 0.9479],
#  [0.7274, 0.7305, 0.8649, 0.9479, 1.0000]]


ImportError: cannot import name 'PreTrainedModel' from 'transformers' (C:\Users\kbpsh\OneDrive\development\project\minirag_dayo\.venv\Lib\site-packages\transformers\__init__.py)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'